In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from scenarios import SCENARIOS
from simulate import simulate_greedy, simulate_mpc
from mpc import MpcSimulationParameters

In [ ]:
# Select scenario

SCENARIO = SCENARIOS[1]
print(f"Running scenario: {SCENARIO.name}")

In [ ]:
# Scenario forecasts

plt.plot(
    SCENARIO.forecasts.price,
    label="Price (£ / kWh)",
    drawstyle="steps-post",
    marker="o",
)
plt.plot(
    SCENARIO.forecasts.demand,
    label="Demand (kWh)",
    drawstyle="steps-post",
    marker="x",
)
plt.legend()

In [ ]:
# Run scenario

results_greedy = simulate_greedy(SCENARIO)
results_mpc = simulate_mpc(SCENARIO, parameters=MpcSimulationParameters(horizon=15))

In [ ]:
# Plot results

fig, axes = plt.subplots(4, 1, figsize=(10, 8), sharex=True)

# Forecasts
axes[0].plot(
    SCENARIO.forecasts.demand[: SCENARIO.timesteps],
    label="Demand (kWh)",
    drawstyle="steps-post",
    marker="o",
    lw=2,
    color="tab:blue",
)
axes[0].plot(
    SCENARIO.forecasts.price[: SCENARIO.timesteps],
    label="Price (£ / kWh)",
    drawstyle="steps-post",
    marker="x",
    lw=2,
    color="tab:orange",
)
axes[0].set(ylabel="Demand / Price")
axes[0].legend(loc=1)

# Grid flows
axes[1].plot(
    np.arange(-1, SCENARIO.timesteps),
    results_greedy.grid_flow,
    label="Greedy",
    lw=2,
    drawstyle="steps-post",
)
axes[1].plot(
    np.arange(-1, SCENARIO.timesteps),
    results_mpc.grid_flow,
    label="MPC",
    lw=2,
    drawstyle="steps-post",
)
axes[1].set(ylabel="Grid Inflow (kWh)")
axes[1].legend(loc=1, ncols=1)

# State of charge
axes[2].plot(
    np.arange(-1, SCENARIO.timesteps),
    results_greedy.state_of_charge,
    label="Greedy",
    lw=2,
    drawstyle="steps-post",
)
axes[2].plot(
    np.arange(-1, SCENARIO.timesteps),
    results_mpc.state_of_charge,
    label="MPC",
    lw=2,
    drawstyle="steps-post",
)
axes[2].set(ylabel="State of Charge (kWh)")
axes[2].legend(loc=4, ncols=1)

# Cost of charge
axes[3].plot(
    np.arange(-1, SCENARIO.timesteps),
    np.cumsum(results_greedy.cost),
    label="Greedy",
    lw=2,
    drawstyle="steps-post",
)
axes[3].plot(
    np.arange(-1, SCENARIO.timesteps),
    np.cumsum(results_mpc.cost),
    label="MPC",
    lw=2,
    drawstyle="steps-post",
)
axes[3].set(ylabel="Cost (£)")
axes[3].legend(loc=4, ncols=1)

for ax in axes:
    ax.grid()
    ax.set(xlim=(-5, SCENARIO.timesteps + 5))

fig.align_labels()
fig.tight_layout()
plt.savefig(SCENARIO.name, dpi=300);